# Silver — Central Bank Indicators

Extracts policy rate and CPI from Bronze, fixes CPI date format, and writes a clean indicators table.

**Source:** `bronze.central_bank_raw`  
**Output:** `silver.central_bank_indicators` (Delta table)  
**Metrics:** `policy_rate` (daily), `cpi` (monthly)  

In [ ]:
from pyspark.sql import functions as F

df = spark.read.format("delta").table("bronze.central_bank_raw")

print(f"Bronze rows: {df.count()}")
df.groupBy("series_name").count().show(truncate=False)

In [ ]:
# Policy rate — main rate series, date already parsed correctly in Bronze
df_policy = (
    df.filter(F.col("series_name") == "Meginvextir (vextir á 7 daga bundnum innlánum)")
      .withColumn("metric", F.lit("policy_rate"))
)

# CPI — Bronze stores dates as string in M/d/yyyy HH:mm:ss a format; reparse explicitly
df_cpi = (
    df.filter(F.col("series_name") == "Vísitala neysluverðs")
      .withColumn("metric", F.lit("cpi"))
      .withColumn("date", F.date_format(
          F.to_timestamp("date", "M/d/yyyy HH:mm:ss a"), "yyyy-MM-dd"
      ))
)

df_silver = (
    df_policy.union(df_cpi)
    .select(
        F.to_date("date").alias("date"),
        "metric",
        "series_name",
        F.round("value", 4).alias("value"),
        F.current_timestamp().alias("processed_at"),
    )
    .orderBy("metric", "date")
)

df_silver.groupBy("metric").count().show()
df_silver.filter(F.col("metric") == "cpi").show(5)

In [ ]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.central_bank_indicators")

print(f"Saved to silver.central_bank_indicators — {df_silver.count()} rows")